# JetBot Collision Avoidance - Gamepad-Driven Data Collection

This notebook merges two things you've already got working separately:

1. **Data collection** from `collision_avoidance/data_collection.ipynb` — a live 224x224 camera feed and snapshot-saving into `dataset/free` and `dataset/blocked`.
2. **Gamepad teleoperation** from `Jetbot_Button_Teleop.ipynb` — D-pad drive, dual-joystick per-wheel control, and bumper speed adjustment.

The idea: drive JetBot around with the controller, and label images **on the fly** without touching the keyboard or mouse — just tap a button the instant the robot is in a `blocked` or `free` situation.

| Control | Action |
|---|---|
| D-pad Up | Drive forward at current speed |
| D-pad Down | Drive backward at current speed |
| D-pad Left | Pivot turn left |
| D-pad Right | Pivot turn right |
| Left joystick (push/pull) | Left motor spins clockwise (push) / counter-clockwise (pull) |
| Right joystick (push/pull) | Right motor spins clockwise (push) / counter-clockwise (pull) |
| Right Bumper (RB) | Increase speed |
| Left Bumper (LB) | Decrease speed |
| **A button** | **Save current frame to `dataset/free`** |
| **B button** | **Save current frame to `dataset/blocked`** |
| X button | Emergency stop + reset speed |




## 1. Initialize the robot, camera, and controller

Go to the gamepad tester website http://hardwaretester.com/gamepad  and note the index number; if needed, edit the “ … widgets.Controller(index= )“ line in the cell. 

In [ ]:
import os
import traitlets
import ipywidgets.widgets as widgets
from IPython.display import display
from jetbot import Robot, Camera, bgr8_to_jpeg
from uuid import uuid1

robot = Robot()

# 224x224 matches what the collision-avoidance training pipeline expects
camera = Camera.instance(width=224, height=224)

image = widgets.Image(format='jpeg', width=224, height=224)
camera_link = traitlets.dlink((camera, 'value'), (image, 'value'), transform=bgr8_to_jpeg)

controller = widgets.Controller(index=0)  # change index if you have more than one gamepad connected
display(controller)


Before continuing: press a couple of buttons on the physical controller. The widget above needs at least one input event before `controller.buttons` / `controller.axes` populate. If it stays blank, make sure the browser has recognized the gamepad.

## 2. (Optional) Find your button and axis indices

Button and axis numbering isn't identical across every controller. Run this cell, then press buttons and move both joysticks on your controller one at a time, and watch the output to confirm which index is which (D-pad, bumpers, A/B/X buttons, left stick Y, right stick Y). Skip this step if you already know your controller's mapping.

In [ ]:
button_output = widgets.Output()
display(button_output)

def on_button_change(change, index):
    if change['new'] > 0:
        with button_output:
            print(f"Button {index} pressed (value={change['new']:.2f})")

def on_axis_change(change, index):
    with button_output:
        print(f"Axis {index} value={change['new']:.2f}")

for i, button in enumerate(controller.buttons):
    button.observe(lambda change, index=i: on_button_change(change, index), names='value')

for i, axis in enumerate(controller.axes):
    axis.observe(lambda change, index=i: on_axis_change(change, index), names='value')

print(f"Watching {len(controller.buttons)} buttons and {len(controller.axes)} axes.")
print("Press each button and move each stick fully up/down to identify indices.")


In [ ]:
# Run this once you're done identifying indices, so the diagnostic
# handlers above don't stick around and clutter behavior later.
for button in controller.buttons:
    button.unobserve_all()
for axis in controller.axes:
    axis.unobserve_all()


## 3. Configure the mapping and speeds

Edit the indices below to match what you found in step 2. Values shown are typical for an Xbox-style gamepad in Chrome. Confirm against your own results — pushing a stick *up/forward* usually reads as a **negative** axis value, so that's the assumption baked into the joystick handler below. If your forward push reads positive instead, flip the sign in `JOY_FORWARD_SIGN`.

In [ ]:
# --- Drive button mapping: adjust to match YOUR controller ---
BTN_FORWARD  = 12   # D-pad Up
BTN_BACKWARD = 13   # D-pad Down
BTN_LEFT     = 14   # D-pad Left
BTN_RIGHT    = 15   # D-pad Right
BTN_SPEED_UP = 5    # Right bumper (RB)
BTN_SPEED_DN = 4    # Left bumper (LB)
BTN_ESTOP    = 2    # X button (moved off A so A is free for data collection)

# --- Data collection button mapping ---
BTN_FREE     = 0    # A button -> save snapshot to dataset/free
BTN_BLOCK    = 1    # B button -> save snapshot to dataset/blocked

# --- Axis mapping: adjust to match YOUR controller ---
AXIS_LEFT_Y  = 1    # Left stick, vertical
AXIS_RIGHT_Y = 3    # Right stick, vertical
JOY_DEADZONE = 0.2  # ignore stick movement smaller than this (drift/noise)
JOY_FORWARD_SIGN = 1  # flip to -1 if your diagnostic showed the opposite

# --- Speed settings ---
DEFAULT_SPEED = 0.4
DEFAULT_TURN_SPEED = 0.35
SPEED_STEP = 0.1
SPEED_MIN = 0.1
SPEED_MAX = 1.0

speed = DEFAULT_SPEED
turn_speed = DEFAULT_TURN_SPEED


## 4. Set up the dataset folders and image counters

Same as the original data collection notebook: a `dataset` folder with `free` and `blocked` subfolders, plus running counts so you can keep the two classes roughly balanced while you drive around.

In [ ]:
blocked_dir = 'dataset/blocked'
free_dir = 'dataset/free'

# we have this "try/except" statement because these next functions can throw an error if the directories exist already
try:
    os.makedirs(free_dir)
    os.makedirs(blocked_dir)
except FileExistsError:
    print('Directories not created because they already exist')

button_layout = widgets.Layout(width='128px', height='64px')
free_count = widgets.IntText(layout=button_layout, value=len(os.listdir(free_dir)), description='free')
blocked_count = widgets.IntText(layout=button_layout, value=len(os.listdir(blocked_dir)), description='blocked')

# On-screen buttons are kept as a backup/manual option alongside the gamepad
free_button = widgets.Button(description='add free', button_style='success', layout=button_layout)
blocked_button = widgets.Button(description='add blocked', button_style='danger', layout=button_layout)


## 5. Wire the D-pad, joysticks, and data-collection buttons

In [ ]:
DIRECTION_BUTTONS = [BTN_FORWARD, BTN_BACKWARD, BTN_LEFT, BTN_RIGHT]

def update_motion(change=None):
    """Reads the current state of the four direction buttons and drives
    the robot accordingly. Called any time one of them is pressed or released."""
    up    = controller.buttons[BTN_FORWARD].value  > 0.5
    down  = controller.buttons[BTN_BACKWARD].value > 0.5
    left  = controller.buttons[BTN_LEFT].value     > 0.5
    right = controller.buttons[BTN_RIGHT].value    > 0.5

    if up and not down:
        robot.forward(speed)
    elif down and not up:
        robot.backward(speed)
    elif left and not right:
        robot.left(turn_speed)
    elif right and not left:
        robot.right(turn_speed)
    else:
        robot.stop()

def joystick_motor_value(axis_value):
    """Maps a raw stick position to a fixed +-speed with a center deadzone,
    rather than a value proportional to how far the stick is pushed."""
    signed_value = axis_value * JOY_FORWARD_SIGN
    if signed_value > JOY_DEADZONE:
        return speed        # pushed forward -> clockwise
    elif signed_value < -JOY_DEADZONE:
        return -speed       # pulled backward -> counter-clockwise
    else:
        return 0.0

def update_left_joystick(change):
    robot.left_motor.value = joystick_motor_value(change['new'])

def update_right_joystick(change):
    robot.right_motor.value = joystick_motor_value(change['new'])

def refresh_joysticks():
    """Re-applies the current stick positions, used after a speed change so
    a stick that's already held over responds to the new speed immediately."""
    robot.left_motor.value = joystick_motor_value(controller.axes[AXIS_LEFT_Y].value)
    robot.right_motor.value = joystick_motor_value(controller.axes[AXIS_RIGHT_Y].value)

def change_speed(change, delta):
    global speed, turn_speed
    if change['new'] > 0.5:  # only act on press, not release
        speed = round(max(SPEED_MIN, min(SPEED_MAX, speed + delta)), 2)
        turn_speed = round(max(SPEED_MIN, min(SPEED_MAX, turn_speed + delta)), 2)
        print(f"Speed: {speed:.2f}   Turn speed: {turn_speed:.2f}")
        update_motion()      # re-apply D-pad state if a direction is held
        refresh_joysticks()  # re-apply joystick state if a stick is held

def emergency_stop(change):
    global speed, turn_speed
    if change['new'] > 0.5:
        robot.stop()
        speed = DEFAULT_SPEED
        turn_speed = DEFAULT_TURN_SPEED
        print("EMERGENCY STOP - speed reset to defaults")

def save_snapshot(directory):
    image_path = os.path.join(directory, str(uuid1()) + '.jpg')
    with open(image_path, 'wb') as f:
        f.write(image.value)

def save_free():
    global free_dir, free_count
    save_snapshot(free_dir)
    free_count.value = len(os.listdir(free_dir))
    print(f"Saved free image ({free_count.value} total)")

def save_blocked():
    global blocked_dir, blocked_count
    save_snapshot(blocked_dir)
    blocked_count.value = len(os.listdir(blocked_dir))
    print(f"Saved blocked image ({blocked_count.value} total)")

def on_block_button(change):
    if change['new'] > 0.5:  # only act on press, not release
        save_blocked()

def on_free_button(change):
    if change['new'] > 0.5:
        save_free()

# Direction buttons
for i in DIRECTION_BUTTONS:
    controller.buttons[i].observe(update_motion, names='value')

# Joysticks
controller.axes[AXIS_LEFT_Y].observe(update_left_joystick, names='value')
controller.axes[AXIS_RIGHT_Y].observe(update_right_joystick, names='value')

# Speed buttons
controller.buttons[BTN_SPEED_UP].observe(lambda c: change_speed(c, SPEED_STEP), names='value')
controller.buttons[BTN_SPEED_DN].observe(lambda c: change_speed(c, -SPEED_STEP), names='value')

# Emergency stop
controller.buttons[BTN_ESTOP].observe(emergency_stop, names='value')

# Data collection buttons
controller.buttons[BTN_BLOCK].observe(on_block_button, names='value')
controller.buttons[BTN_FREE].observe(on_free_button, names='value')

# Manual on-screen buttons (backup, uses the same save functions)
free_button.on_click(lambda x: save_free())
blocked_button.on_click(lambda x: save_blocked())

print("Controller wired up:")
print("  D-pad          -> drive / turn (both wheels together)")
print("  Left stick     -> left motor (push=clockwise, pull=counter-clockwise)")
print("  Right stick    -> right motor (push=clockwise, pull=counter-clockwise)")
print("  RB / LB        -> speed up / down")
print("  X              -> emergency stop")
print("  A              -> save 'free' image")
print("  B              -> save 'blocked' image")
print(f"Starting speed: {speed:.2f}   Starting turn speed: {turn_speed:.2f}")


## 6. Drive around and collect data

Display the live feed and counters below, then drive JetBot with the D-pad or joysticks. When it's clear to move forward, tap **A** to save a `free` frame; the instant the robot is in a scenario where its safety bubble is violated, tap **B** to save a `blocked` frame. The on-screen buttons below do the same thing if you'd rather click.

Tips carried over from the original notebook:

1. Try different orientations
2. Try different lighting
3. Try varied object / collision types: walls, ledges, objects
4. Try different textured floors / objects: patterned, smooth, glass, etc.

Aim for roughly balanced counts of `free` and `blocked`, and at least ~100 images per class as a starting point.

In [ ]:
display(image)
display(widgets.HBox([free_count, free_button]))
display(widgets.HBox([blocked_count, blocked_button]))


## 7. Disconnect when finished

Run this before closing the notebook or restarting the kernel, so the robot doesn't keep responding to a controller that's no longer being watched, and so the camera is released for the next notebook (e.g. `train_model.ipynb` or a live demo notebook).

In [ ]:
for i in DIRECTION_BUTTONS + [BTN_SPEED_UP, BTN_SPEED_DN, BTN_ESTOP, BTN_BLOCK, BTN_FREE]:
    controller.buttons[i].unobserve_all()
controller.axes[AXIS_LEFT_Y].unobserve_all()
controller.axes[AXIS_RIGHT_Y].unobserve_all()
robot.stop()
camera.stop()
print("Disconnected. Robot stopped, camera released.")


## 8. Package the dataset

Once you've collected enough data, zip it up for transfer to the GPU machine running `collision_avoidance/train_model.ipynb`.

In [ ]:
!zip -r -q dataset.zip dataset
